In [3]:
# ==========================================================
# 🚀 INDIAN PINES → CNN → TM → FULL PIPELINE (121/class)
# FINAL (STABLE + PUBLICATION READY)
# ==========================================================

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix
from matplotlib.tri import Triangulation

from tmu.models.classification.vanilla_classifier import TMClassifier

# ================= REPRODUCIBILITY =================
np.random.seed(42)
torch.manual_seed(42)

# ================= SETTINGS =================
RESULTS_DIR = "results_ip_121_final"
os.makedirs(RESULTS_DIR, exist_ok=True)

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 14
})
sns.set_style("whitegrid")

DEVICE = torch.device("cpu")

# ================= LOAD =================
X = np.load("indianpinearray.npy")
y = np.load("IPgt.npy")

# ================= PATCH EXTRACTION =================
PATCH_SIZE = 9

def create_patches(X, y, patch_size):
    margin = patch_size // 2
    padded = np.pad(X, ((margin,margin),(margin,margin),(0,0)), mode='reflect')

    patches, labels = [], []

    for i in range(margin, X.shape[0] + margin):
        for j in range(margin, X.shape[1] + margin):

            label = y[i-margin, j-margin]
            if label == 0:
                continue

            patch = padded[i-margin:i+margin+1, j-margin:j+margin+1]
            patches.append(patch.transpose(2,0,1))
            labels.append(label - 1)

    return np.array(patches), np.array(labels)

print("Creating patches...")
X_patches, y_patches = create_patches(X, y, PATCH_SIZE)

# ================= 121 PER CLASS =================
def stratified_fixed_split(X, y, samples_per_class=121):

    X_train, y_train = [], []
    X_test, y_test = [], []

    for c in np.unique(y):
        idx = np.where(y == c)[0]
        np.random.shuffle(idx)

        n_train = min(samples_per_class, len(idx)//2)

        train_idx = idx[:n_train]
        test_idx  = idx[n_train:]

        X_train.append(X[train_idx])
        y_train.append(y[train_idx])

        X_test.append(X[test_idx])
        y_test.append(y[test_idx])

    return (
        np.concatenate(X_train),
        np.concatenate(X_test),
        np.concatenate(y_train),
        np.concatenate(y_test)
    )

print("Applying 121/class split...")
X_train, X_test, y_train, y_test = stratified_fixed_split(X_patches, y_patches, 121)

num_classes = len(np.unique(y_train))

# ================= DATASET =================
class HSIDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx])

train_loader = DataLoader(HSIDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader  = DataLoader(HSIDataset(X_test, y_test), batch_size=32)

# ================= CNN =================
class CNN(nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, 64, 3, padding=1)
        self.bn = nn.BatchNorm2d(64)

        self.conv2 = nn.Conv2d(64,128,3,padding=1)
        self.conv3 = nn.Conv2d(128,256,3,padding=1)

    def forward(self,x):
        x = F.relu(self.bn(self.conv1(x)))
        x = F.max_pool2d(x,2)
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        return x

model = CNN(X.shape[2]).to(DEVICE)
classifier = nn.Linear(256, num_classes).to(DEVICE)

optimizer = torch.optim.Adam(list(model.parameters())+list(classifier.parameters()), lr=0.0003)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print("Training CNN...")
for epoch in range(40):
    model.train()
    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        feat = model(x)
        pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
        loss = criterion(classifier(pooled), y)
        loss.backward()
        optimizer.step()

print("CNN Done")

# ================= FEATURE EXTRACTION =================
def extract(loader):
    X_out, y_out = [], []
    model.eval()

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            feat = model(xb)

            batch = []
            for sample in feat:
                f = []
                for ch in sample:
                    f.append(ch.mean().item() > 0.3)
                    f.append(ch.max().item() > 0.6)
                batch.append(f)

            X_out.append(np.array(batch))
            y_out.extend(yb.numpy())

    return (
        np.vstack(X_out).astype(np.uint32),
        np.array(y_out).astype(np.uint32)
    )

print("Extracting features...")
X_train_tm, y_train_tm = extract(train_loader)
X_test_tm, y_test_tm   = extract(test_loader)

# ================= SPECIFICITY =================
def compute_specificity(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    specs = []

    for i in range(len(cm)):
        TP = cm[i,i]
        FP = cm[:,i].sum() - TP
        TN = cm.sum() - (cm[i,:].sum() + FP)
        specs.append(TN/(TN+FP+1e-9))

    return np.mean(specs)

# ================= OPTUNA =================
def objective(trial):

    tm = TMClassifier(
        number_of_clauses=trial.suggest_int("num_clauses",2000,8000,500),
        T=trial.suggest_int("T",100,2000,100),
        s=trial.suggest_float("s",2.0,8.0),
        platform="CPU",
        weighted_clauses=True
    )

    tm.fit(X_train_tm, y_train_tm, epochs=1)
    preds = tm.predict(X_test_tm)

    return accuracy_score(y_test_tm, preds)

print("Running Optuna...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

best_params = study.best_params
print("Best Params:", best_params)

# ================= FINAL TM =================
tm = TMClassifier(
    number_of_clauses=best_params["num_clauses"],
    T=best_params["T"],
    s=best_params["s"],
    platform="CPU",
    weighted_clauses=True
)

acc_list, spec_list = [], []

for epoch in range(60):
    tm.fit(X_train_tm, y_train_tm, epochs=1)
    preds = tm.predict(X_test_tm)

    acc_list.append(accuracy_score(y_test_tm, preds))
    spec_list.append(compute_specificity(y_test_tm, preds))

# ================= CONFUSION MATRIX =================
cm = confusion_matrix(y_test_tm, preds)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(6,6))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Normalized Confusion Matrix")
plt.savefig(os.path.join(RESULTS_DIR,"cm.png"), dpi=600)
plt.close()

# ================= PER CLASS =================
per_class = cm.diagonal() / cm.sum(axis=1)

plt.figure(figsize=(6,6))
plt.bar(range(len(per_class)), per_class)
plt.xlabel("Class")
plt.ylabel("Accuracy")
plt.title("Per-Class Accuracy")
plt.savefig(os.path.join(RESULTS_DIR,"per_class.png"), dpi=600)
plt.close()

# ================= CONVERGENCE =================
plt.figure(figsize=(6,6))
plt.plot(acc_list,label="Accuracy")
plt.plot(spec_list,label="Specificity")
plt.legend()
plt.title("Training Convergence")
plt.savefig(os.path.join(RESULTS_DIR,"convergence.png"), dpi=600)
plt.close()

# ================= OPTUNA =================
best_vals=[]
best=0
for t in study.trials:
    best=max(best,t.value)
    best_vals.append(best)

plt.figure(figsize=(6,6))
plt.plot(best_vals)
plt.title("Optuna Optimization Progress")
plt.savefig(os.path.join(RESULTS_DIR,"optuna.png"), dpi=600)
plt.close()

# ================= CONTOUR (FIXED) =================
points_T, points_cls, acc_vals = [], [], []

for clauses in range(200, 1501, 150):

    if clauses % 2 != 0:
        clauses += 1

    for T_val in range(50, 1101, 100):

        tm_temp = TMClassifier(
            number_of_clauses=clauses,
            T=T_val,
            s=best_params["s"],
            platform="CPU",
            weighted_clauses=True
        )

        try:
            tm_temp.fit(X_train_tm, y_train_tm, epochs=3)
            preds = tm_temp.predict(X_test_tm)

            acc = accuracy_score(y_test_tm, preds) * 100

            acc_vals.append(acc)
            points_T.append(T_val)
            points_cls.append(clauses)

        except:
            continue

tri = Triangulation(points_T, points_cls)

plt.figure(figsize=(6,6))
plt.tricontourf(tri, acc_vals, levels=20, cmap="viridis")
plt.colorbar(label="Accuracy (%)")
plt.xlabel("Threshold (T)")
plt.ylabel("Clauses")
plt.title("TM Hyperparameter Landscape")
plt.savefig(os.path.join(RESULTS_DIR,"contour.png"), dpi=600)
plt.close()

print("✅ FULL PIPELINE COMPLETE (STABLE + PUBLICATION READY)")

Creating patches...
Applying 121/class split...
Training CNN...
CNN Done
Extracting features...
Running Optuna...


/tmp/ipykernel_2590906/2723639198.py:199: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  number_of_clauses=trial.suggest_int("num_clauses",2000,8000,500),
/tmp/ipykernel_2590906/2723639198.py:200: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/

Best Params: {'num_clauses': 6500, 'T': 1000, 's': 2.1560504772654867}
✅ FULL PIPELINE COMPLETE (STABLE + PUBLICATION READY)


In [2]:
# ==========================================================
# 🚀 SINGLE FOLDER → CNN → TM → FULL PUBLICATION PIPELINE
# (FIXED VERSION — NO CLAUSE MISMATCH)
# ==========================================================

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
import matplotlib.pyplot as plt
import seaborn as sns

from glob import glob
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix
from matplotlib.tri import Triangulation

from tmu.models.classification.vanilla_classifier import TMClassifier

# ================= REPRODUCIBILITY =================
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

# ================= SETTINGS =================
BASE_DIR = "sample_dataset"
TRAIN_PER_CLASS = 121
IMAGE_SIZE = 64
BATCH_SIZE = 32
CNN_EPOCHS = 40

DEVICE = torch.device("cpu")

RESULTS_DIR = "results_final_SAMPLE_Dataset"
os.makedirs(RESULTS_DIR, exist_ok=True)

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 14
})
sns.set_style("whitegrid")

# ================= DATA PREP =================
train_samples, test_samples = [], []

classes = sorted(os.listdir(BASE_DIR))

for label, cname in enumerate(classes):
    paths = glob(os.path.join(BASE_DIR, cname, "*"))
    random.shuffle(paths)

    train = paths[:TRAIN_PER_CLASS]
    test  = paths[TRAIN_PER_CLASS:]

    train_samples += [(p, label) for p in train]
    test_samples  += [(p, label) for p in test]

print(f"Train: {len(train_samples)}, Test: {len(test_samples)}")

# ================= DATASET =================
class ImageDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path)
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2,0,1))
        return torch.tensor(img), torch.tensor(label)

train_loader = DataLoader(ImageDataset(train_samples), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(ImageDataset(test_samples), batch_size=BATCH_SIZE)

num_classes = len(classes)

# ================= CNN =================
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv3 = nn.Conv2d(3,16,3,padding=1)
        self.conv5 = nn.Conv2d(3,16,5,padding=2)
        self.conv7 = nn.Conv2d(3,16,7,padding=3)

        self.block1 = nn.Sequential(
            nn.Conv2d(48,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(256,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

    def forward(self,x):
        x = torch.cat([
            F.relu(self.conv3(x)),
            F.relu(self.conv5(x)),
            F.relu(self.conv7(x))
        ], dim=1)

        return self.block4(self.block3(self.block2(self.block1(x))))

model = CNN().to(DEVICE)
classifier = nn.Linear(256, num_classes).to(DEVICE)

optimizer = torch.optim.Adam(list(model.parameters())+list(classifier.parameters()), lr=0.0003)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# ================= TRAIN CNN =================
print("Training CNN...")
for epoch in range(CNN_EPOCHS):
    model.train()
    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        feat = model(x)
        pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
        loss = criterion(classifier(pooled), y)
        loss.backward()
        optimizer.step()

print("CNN Done")

# ================= FEATURE EXTRACTION =================
def extract(loader):
    X,y = [],[]
    model.eval()

    with torch.no_grad():
        for xb,yb in loader:
            xb = xb.to(DEVICE)
            feat = model(xb)

            batch=[]
            for sample in feat:
                f=[]
                for ch in sample:
                    f.append(ch.mean().item()>0.3)
                    f.append(ch.max().item()>0.6)
                batch.append(f)

            X.append(np.array(batch))
            y.extend(yb.numpy())

    return np.vstack(X).astype(np.uint32), np.array(y).astype(np.uint32)

X_train_tm, y_train_tm = extract(train_loader)
X_test_tm, y_test_tm   = extract(test_loader)

# ================= SPECIFICITY =================
def compute_specificity(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    specs=[]
    for i in range(len(cm)):
        TP=cm[i,i]
        FP=cm[:,i].sum()-TP
        TN=cm.sum()-(cm[i,:].sum()+FP)
        specs.append(TN/(TN+FP+1e-9))
    return np.mean(specs)

# ================= OPTUNA =================
def objective(trial):
    tm = TMClassifier(
        number_of_clauses=trial.suggest_int("num_clauses",2000,8000,500),
        T=trial.suggest_int("T",100,2000,100),
        s=trial.suggest_float("s",2.0,8.0),
        platform="CPU",
        weighted_clauses=True
    )

    tm.fit(X_train_tm, y_train_tm, epochs=1)
    preds = tm.predict(X_test_tm)
    return accuracy_score(y_test_tm, preds)

print("Running Optuna...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

best_params = study.best_params
print("Best:", best_params)

# ================= FINAL TM =================
tm = TMClassifier(
    number_of_clauses=best_params["num_clauses"],
    T=best_params["T"],
    s=best_params["s"],
    platform="CPU",
    weighted_clauses=True
)

acc_list, spec_list = [],[]

for epoch in range(60):
    tm.fit(X_train_tm, y_train_tm, epochs=1)
    preds = tm.predict(X_test_tm)

    acc_list.append(accuracy_score(y_test_tm,preds))
    spec_list.append(compute_specificity(y_test_tm,preds))

# ================= CONFUSION MATRIX =================
cm = confusion_matrix(y_test_tm, preds)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(6,6))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Normalized Confusion Matrix")
plt.savefig(os.path.join(RESULTS_DIR,"cm.png"), dpi=600)
plt.close()

# ================= PER CLASS =================
per_class = cm.diagonal()/cm.sum(axis=1)

plt.figure(figsize=(6,6))
plt.bar(range(len(per_class)), per_class)
plt.xlabel("Class")
plt.ylabel("Accuracy")
plt.title("Per-Class Accuracy")
plt.savefig(os.path.join(RESULTS_DIR,"per_class.png"), dpi=600)
plt.close()

# ================= CONVERGENCE =================
plt.figure(figsize=(6,6))
plt.plot(acc_list,label="Accuracy")
plt.plot(spec_list,label="Specificity")
plt.legend()
plt.title("Training Convergence")
plt.savefig(os.path.join(RESULTS_DIR,"convergence.png"), dpi=600)
plt.close()

# ================= OPTUNA =================
best_vals=[]
best=0
for t in study.trials:
    best=max(best,t.value)
    best_vals.append(best)

plt.figure(figsize=(6,6))
plt.plot(best_vals)
plt.title("Optuna Optimization Progress")
plt.savefig(os.path.join(RESULTS_DIR,"optuna.png"), dpi=600)
plt.close()

# ================= CONTOUR (FIXED) =================
points_T, points_cls, acc_vals = [],[],[]

for clauses in range(200, 1501, 150):

    if clauses % 2 != 0:
        clauses += 1   # 🔥 critical fix

    for T_val in range(50, 1101, 100):

        tm_temp = TMClassifier(
            number_of_clauses=clauses,
            T=T_val,
            s=best_params["s"],
            platform="CPU",
            weighted_clauses=True
        )

        try:
            tm_temp.fit(X_train_tm, y_train_tm, epochs=3)
            preds = tm_temp.predict(X_test_tm)

            acc_vals.append(accuracy_score(y_test_tm,preds)*100)
            points_T.append(T_val)
            points_cls.append(clauses)

        except:
            continue

tri = Triangulation(points_T, points_cls)

plt.figure(figsize=(6,6))
plt.tricontourf(tri, acc_vals, levels=20, cmap="viridis")
plt.colorbar(label="Accuracy (%)")
plt.xlabel("Threshold (T)")
plt.ylabel("Clauses")
plt.title("TM Hyperparameter Landscape")
plt.savefig(os.path.join(RESULTS_DIR,"contour.png"), dpi=600)
plt.close()

print("✅ ALL RESULTS GENERATED (STABLE + PUBLICATION READY)")

Train: 1154, Test: 191
Training CNN...
CNN Done
Running Optuna...


/tmp/ipykernel_2590906/2478439036.py:192: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  number_of_clauses=trial.suggest_int("num_clauses",2000,8000,500),
/tmp/ipykernel_2590906/2478439036.py:193: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/

Best: {'num_clauses': 7500, 'T': 300, 's': 7.761303192872338}
✅ ALL RESULTS GENERATED (STABLE + PUBLICATION READY)


In [5]:
# ==========================================================
# 🚀 EUROSAT → CNN → TM → OPTUNA → FINAL STABLE PIPELINE
# ==========================================================

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
import matplotlib.pyplot as plt
import seaborn as sns

from glob import glob
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from matplotlib.tri import Triangulation

from tmu.models.classification.vanilla_classifier import TMClassifier

# ================= SETTINGS =================
BASE_DIR = "EuroSAT"
TRAIN_PER_CLASS = 121
IMAGE_SIZE = 64
BATCH_SIZE = 32
CNN_EPOCHS = 40

DEVICE = torch.device("cpu")  # 🔥 stable only

RESULTS_DIR = "results_final_safe"
os.makedirs(RESULTS_DIR, exist_ok=True)

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 14
})
sns.set_style("whitegrid")

# ================= DATA =================
train_samples, test_samples = [], []
classes = sorted(os.listdir(BASE_DIR))

for label, cname in enumerate(classes):
    paths = glob(os.path.join(BASE_DIR, cname, "*"))
    random.shuffle(paths)

    train = paths[:TRAIN_PER_CLASS]
    test  = paths[TRAIN_PER_CLASS:]

    train_samples += [(p, label) for p in train]
    test_samples  += [(p, label) for p in test]

# ================= DATASET =================
class EuroSATDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = cv2.imread(path, cv2.IMREAD_COLOR)
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2,0,1))

        return torch.tensor(img), torch.tensor(label)

train_loader = DataLoader(EuroSATDataset(train_samples), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(EuroSATDataset(test_samples), batch_size=BATCH_SIZE)

num_classes = len(classes)

# ================= CNN =================
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv3 = nn.Conv2d(3,16,3,padding=1)
        self.conv5 = nn.Conv2d(3,16,5,padding=2)
        self.conv7 = nn.Conv2d(3,16,7,padding=3)

        self.block1 = nn.Sequential(nn.Conv2d(48,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.block2 = nn.Sequential(nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2))
        self.block3 = nn.Sequential(nn.Conv2d(128,256,3,padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2))
        self.block4 = nn.Sequential(nn.Conv2d(256,256,3,padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.Dropout(0.4))

    def forward(self,x):
        x = torch.cat([F.relu(self.conv3(x)),F.relu(self.conv5(x)),F.relu(self.conv7(x))], dim=1)
        return self.block4(self.block3(self.block2(self.block1(x))))

model = CNN().to(DEVICE)
classifier = nn.Linear(256, num_classes).to(DEVICE)

optimizer = torch.optim.Adam(list(model.parameters())+list(classifier.parameters()), lr=0.0003)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# ================= TRAIN CNN =================
print("Training CNN...")
for epoch in range(CNN_EPOCHS):
    model.train()
    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        feat = model(x)
        pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
        loss = criterion(classifier(pooled), y)
        loss.backward()
        optimizer.step()

print("CNN Done")

# ================= FEATURE EXTRACTION =================
def extract(loader):
    X,y = [],[]
    model.eval()

    with torch.no_grad():
        for xb,yb in loader:
            xb = xb.to(DEVICE)
            feat = model(xb)

            batch=[]
            for sample in feat:
                f=[]
                for ch in sample:
                    f.append(ch.mean().item()>0.3)
                    f.append(ch.max().item()>0.6)
                batch.append(f)

            X.append(np.array(batch))
            y.extend(yb.numpy())

    return np.vstack(X).astype(np.uint32), np.array(y).astype(np.uint32)

X_all, y_all = extract(train_loader)
X_test_tm, y_test_tm = extract(test_loader)

# ================= SPLIT =================
X_train_tm, X_val_tm, y_train_tm, y_val_tm = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=42
)

# ================= OPTUNA =================
def objective(trial):

    clauses = trial.suggest_int("num_clauses", low=2000, high=8000, step=500)
    clauses = (clauses // 2) * 2   # 🔥 enforce even

    tm = TMClassifier(
        number_of_clauses=clauses,
        T=trial.suggest_int("T", low=100, high=2000, step=100),
        s=trial.suggest_float("s", 2.0, 8.0),
        platform="CPU",
        weighted_clauses=True
    )

    tm.fit(X_train_tm, y_train_tm, epochs=1)
    preds = tm.predict(X_val_tm)

    return accuracy_score(y_val_tm, preds)

print("Running Optuna...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

best_params = study.best_params
best_clauses = (best_params["num_clauses"] // 2) * 2

# ================= FINAL TM =================
tm = TMClassifier(
    number_of_clauses=best_clauses,
    T=best_params["T"],
    s=best_params["s"],
    platform="CPU",
    weighted_clauses=True
)

acc_list = []

for epoch in range(60):
    tm.fit(X_train_tm, y_train_tm, epochs=1)
    preds = tm.predict(X_test_tm)
    acc_list.append(accuracy_score(y_test_tm, preds))

# ================= CONFUSION MATRIX (FIXED) =================
cm = confusion_matrix(y_test_tm, preds)

# Normalize safely (avoid division by zero)
cm_norm = cm.astype(np.float32) / (cm.sum(axis=1, keepdims=True) + 1e-8)

labels = classes  # class names from dataset

plt.figure(figsize=(10,8))

sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels,
    cbar_kws={"label": "Normalized Accuracy"}
)

plt.xlabel("Predicted Label", fontsize=14)
plt.ylabel("True Label", fontsize=14)
plt.title("Normalized Confusion Matrix (EuroSAT)", fontsize=16)

plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()

plt.savefig(os.path.join(RESULTS_DIR, "confusion_matrix_publication.png"), dpi=600)
plt.close()
# ================= CONVERGENCE =================
plt.figure(figsize=(6,6))
plt.plot(acc_list)
plt.title("Accuracy Convergence")
plt.savefig(os.path.join(RESULTS_DIR,"convergence.png"), dpi=600)
plt.close()

# ================= OPTUNA =================
best_vals=[]
best=0
for t in study.trials:
    best=max(best,t.value)
    best_vals.append(best)

plt.figure(figsize=(6,6))
plt.plot(best_vals)
plt.title("Optuna Convergence")
plt.savefig(os.path.join(RESULTS_DIR,"optuna.png"), dpi=600)
plt.close()

# ================= SAFE CONTOUR =================
points_T, points_cls, acc_vals = [],[],[]

for clauses in np.linspace(200,1500,10):
    clauses = int(clauses)
    clauses = (clauses // 2) * 2

    for T_val in np.linspace(50,1100,10):
        try:
            tm_temp = TMClassifier(
                number_of_clauses=clauses,
                T=int(T_val),
                s=best_params["s"],
                platform="CPU",
                weighted_clauses=True
            )

            tm_temp.fit(X_train_tm, y_train_tm, epochs=3)
            preds = tm_temp.predict(X_test_tm)

            acc_vals.append(accuracy_score(y_test_tm,preds)*100)
            points_T.append(T_val)
            points_cls.append(clauses)

        except:
            continue

tri = Triangulation(points_T, points_cls)

plt.figure(figsize=(6,6))
plt.tricontourf(tri, acc_vals, levels=20, cmap="viridis")
plt.colorbar(label="Accuracy (%)")
plt.xlabel("Threshold")
plt.ylabel("Clauses")
plt.savefig(os.path.join(RESULTS_DIR,"contour.png"), dpi=600)
plt.close()

print("✅ FINAL STABLE PIPELINE COMPLETE")

Training CNN...
CNN Done
Running Optuna...
✅ FINAL STABLE PIPELINE COMPLETE


In [1]:
# ==========================================================
# 🚀 CNN → TM → INTERPRETATION (FINAL THESIS VERSION)
# ==========================================================

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from glob import glob
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from tmu.models.classification.vanilla_classifier import TMClassifier

# ================= CONFIG =================

BASE_DIR = "MSTAR-10-Classes"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMAGE_SIZE = 64
TRAIN_PER_CLASS = 121

BATCH_SIZE = 32
CNN_EPOCHS = 40

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLAUSES = 4000
T = 2500
S = 6.0
TM_EPOCHS = 50

classes = sorted(os.listdir(TRAIN_DIR))
num_classes = len(classes)

# ================= DATASET =================

class SARDataset(Dataset):
    def __init__(self, folder, limit=None):
        self.samples = []
        self.classes = sorted(os.listdir(folder))

        for label, cname in enumerate(self.classes):
            paths = glob(os.path.join(folder, cname, "*"))

            if limit:
                paths = random.sample(paths, min(limit, len(paths)))

            for p in paths:
                self.samples.append((p, label))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = img.astype(np.float32) / 255.0

        img = np.stack([img]*3, axis=0)

        return torch.tensor(img), torch.tensor(label)

train_dataset = SARDataset(TRAIN_DIR, TRAIN_PER_CLASS)
test_dataset  = SARDataset(TEST_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# ================= CNN =================

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv3 = nn.Conv2d(3,16,3,padding=1)
        self.conv5 = nn.Conv2d(3,16,5,padding=2)
        self.conv7 = nn.Conv2d(3,16,7,padding=3)

        self.block1 = nn.Sequential(
            nn.Conv2d(48,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(256,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

    def forward(self,x):
        x = torch.cat([
            F.relu(self.conv3(x)),
            F.relu(self.conv5(x)),
            F.relu(self.conv7(x))
        ], dim=1)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        return x

model = CNN().to(DEVICE)
classifier = nn.Linear(256, num_classes).to(DEVICE)

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(classifier.parameters()),
    lr=0.0003
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# ================= TRAIN CNN =================

print("Training CNN...")

for epoch in range(CNN_EPOCHS):
    model.train()
    classifier.train()

    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()

        feat = model(x)
        pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
        out = classifier(pooled)

        loss = criterion(out,y)
        loss.backward()
        optimizer.step()

print("✅ CNN Done")

# ================= FEATURE EXTRACTION =================

feature_map_info = []
built = False

def extract(loader):
    global built
    X, y = [], []

    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            feat = model(xb)

            batch_feats = []

            for sample in feat:
                sample_features = []
                ch_idx = 0

                for ch in sample:
                    ch_np = ch.cpu().numpy()
                    ch_mean = ch_np.mean()
                    ch_std  = ch_np.std() + 1e-6

                    PATCH_GRID = 4
                    h, w = ch_np.shape
                    ph, pw = h//PATCH_GRID, w//PATCH_GRID

                    for i in range(PATCH_GRID):
                        for j in range(PATCH_GRID):

                            patch = ch_np[i*ph:(i+1)*ph, j*pw:(j+1)*pw]

                            mean = patch.mean()
                            std  = patch.std()
                            mx   = patch.max()

                            mean_rel = mean / (ch_mean + 1e-6)
                            std_rel  = std / ch_std
                            cv       = std / (mean + 1e-6)

                            f = [
                                int(mean_rel > 0.8),
                                int(mean_rel > 1.0),
                                int(mean_rel > 1.2),
                                int(std_rel > 0.8),
                                int(std_rel > 1.0),
                                int(cv > 0.5),
                                int(cv > 1.0),
                                int(mx > ch_mean),
                                int(mx > ch_mean + ch_std)
                            ]

                            sample_features.extend(f)

                            if not built:
                                types = ["mean","mean","mean","std","std","cv","cv","max","max"]
                                for t in types:
                                    feature_map_info.append({
                                        "channel": ch_idx,
                                        "pos": (i,j),
                                        "type": t
                                    })

                    ch_idx += 1

                batch_feats.append(sample_features)

            built = True
            X.append(np.array(batch_feats))
            y.extend(yb.numpy())

    return np.vstack(X).astype(np.uint32), np.array(y, dtype=np.uint32)

print("Extracting features...")
X_train, y_train = extract(train_loader)
X_test, y_test   = extract(test_loader)

# ================= TM =================

tm = TMClassifier(
    number_of_clauses=CLAUSES,
    T=T,
    s=S,
    weighted_clauses=True,
    feature_negation=True,
    platform="CPU"
)

print("Training TM...")

for epoch in range(TM_EPOCHS):
    tm.fit(X_train, y_train, epochs=1)
    preds = tm.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"Epoch {epoch+1} → {acc:.4f}")

print("🔥 FINAL ACC:", acc)

# ================= SAMPLE SELECTION =================

preds = tm.predict(X_test)

clause_outputs = tm.transform(X_test).reshape(
    X_test.shape[0], num_classes, tm.number_of_clauses
)

scores = clause_outputs.sum(axis=2)

correct, wrong, ambiguous = [], [], []

for i in range(len(X_test)):
    pred = preds[i]
    true = y_test[i]

    sorted_scores = np.sort(scores[i])
    margin = sorted_scores[-1] - sorted_scores[-2]

    if pred == true:
        correct.append((i, margin))
    else:
        wrong.append((i, margin))

    if margin < 5:
        ambiguous.append((i, margin))

correct = sorted(correct, key=lambda x: -x[1])[:3]
wrong = sorted(wrong, key=lambda x: x[1])[:3]
ambiguous = sorted(ambiguous, key=lambda x: x[1])[:3]

# ================= HEATMAP =================

def generate_heatmap(idx, tag):
    os.makedirs("outputs", exist_ok=True)

    pred = preds[idx]
    true = y_test[idx]

    pred_name = classes[pred]
    true_name = classes[true]

    print(f"[{tag.upper()}] idx={idx} → Pred: {pred_name} | True: {true_name}")

    img, _ = test_dataset[idx]
    img_np = img[0].numpy()

    heatmap = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)

    active = np.where(X_test[idx] == 1)[0]

    PATCH_GRID = 4
    ph = IMAGE_SIZE // PATCH_GRID
    pw = IMAGE_SIZE // PATCH_GRID

    for f in active:
        if f >= len(feature_map_info): continue

        i,j = feature_map_info[f]["pos"]

        patch = img_np[i*ph:(i+1)*ph, j*pw:(j+1)*pw]
        if patch.size == 0: continue

        patch = patch / (patch.max() + 1e-6)
        weight = 2.0 if feature_map_info[f]["type"]=="max" else 1.0

        heatmap[i*ph:(i+1)*ph, j*pw:(j+1)*pw] += patch * weight

    heatmap = cv2.GaussianBlur(heatmap,(9,9),0)
    heatmap = cv2.normalize(heatmap,None,0,255,cv2.NORM_MINMAX).astype(np.uint8)

    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    img_color = cv2.cvtColor((img_np*255).astype(np.uint8), cv2.COLOR_GRAY2BGR)
    overlay = cv2.addWeighted(img_color,0.6,heatmap_color,0.4,0)

    cv2.imwrite(f"outputs/{tag}_{idx}_overlay_mstar.png", overlay)

# ================= RUN =================

print("\nGenerating interpretation...")

for i,_ in correct:
    generate_heatmap(i,"correct")

for i,_ in wrong:
    generate_heatmap(i,"wrong")

for i,_ in ambiguous:
    generate_heatmap(i,"ambiguous")

Training CNN...
✅ CNN Done
Extracting features...
Training TM...
Epoch 1 → 0.5188
Epoch 2 → 0.9600
Epoch 3 → 0.9744
Epoch 4 → 0.9831
Epoch 5 → 0.9802
Epoch 6 → 0.9831
Epoch 7 → 0.9819
Epoch 8 → 0.9843
Epoch 9 → 0.9847
Epoch 10 → 0.9847
Epoch 11 → 0.9852
Epoch 12 → 0.9843
Epoch 13 → 0.9868
Epoch 14 → 0.9868
Epoch 15 → 0.9868
Epoch 16 → 0.9856
Epoch 17 → 0.9856
Epoch 18 → 0.9856
Epoch 19 → 0.9864
Epoch 20 → 0.9872
Epoch 21 → 0.9868
Epoch 22 → 0.9868
Epoch 23 → 0.9868
Epoch 24 → 0.9868
Epoch 25 → 0.9880
Epoch 26 → 0.9880
Epoch 27 → 0.9880
Epoch 28 → 0.9872
Epoch 29 → 0.9872
Epoch 30 → 0.9868
Epoch 31 → 0.9868
Epoch 32 → 0.9880
Epoch 33 → 0.9872
Epoch 34 → 0.9872
Epoch 35 → 0.9868
Epoch 36 → 0.9876
Epoch 37 → 0.9872
Epoch 38 → 0.9872
Epoch 39 → 0.9872
Epoch 40 → 0.9880
Epoch 41 → 0.9880
Epoch 42 → 0.9876
Epoch 43 → 0.9872
Epoch 44 → 0.9872
Epoch 45 → 0.9864
Epoch 46 → 0.9868
Epoch 47 → 0.9872
Epoch 48 → 0.9880
Epoch 49 → 0.9872
Epoch 50 → 0.9880
🔥 FINAL ACC: 0.9880412371134021


/tmp/ipykernel_2674201/1370692012.py:288: RuntimeWarning: overflow encountered in scalar negative
  correct = sorted(correct, key=lambda x: -x[1])[:3]



Generating interpretation...
[CORRECT] idx=122 → Pred: 2S1 | True: 2S1
[CORRECT] idx=261 → Pred: 2S1 | True: 2S1
[CORRECT] idx=450 → Pred: BMP2 | True: BMP2
[WRONG] idx=172 → Pred: ZIL131 | True: 2S1
[WRONG] idx=10 → Pred: ZIL131 | True: 2S1
[WRONG] idx=330 → Pred: BTR70 | True: BMP2
[AMBIGUOUS] idx=122 → Pred: 2S1 | True: 2S1
[AMBIGUOUS] idx=261 → Pred: 2S1 | True: 2S1
[AMBIGUOUS] idx=450 → Pred: BMP2 | True: BMP2


In [2]:
# ==========================================================
# 🚀 INDIAN PINES → CNN → TM → INTERPRETATION (FINAL)
# ==========================================================

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from tmu.models.classification.vanilla_classifier import TMClassifier
import pandas as pd

# ================= CONFIG =================

PATCH_SIZE = 11
TRAIN_PER_CLASS = 200
BATCH_SIZE = 32
CNN_EPOCHS = 40

CLAUSES = 4000
T = 2500
S = 6.0
TM_EPOCHS = 50

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= LOAD DATA =================

print("Loading Indian Pines dataset...")

X = np.load("indianpinearray.npy")   # (145,145,200)
y = np.load("IPgt.npy")              # (145,145)

# ================= PREPROCESS =================

X_gray = np.mean(X, axis=2)
X_gray = (X_gray - X_gray.min()) / (X_gray.max() + 1e-6)
X = np.stack([X_gray]*3, axis=2)

# ================= PATCH EXTRACTION =================

def create_samples(X, y, patch_size, limit_per_class):
    margin = patch_size // 2
    padded = np.pad(X, ((margin,margin),(margin,margin),(0,0)), mode='reflect')

    class_samples = {}

    for i in range(margin, X.shape[0] + margin):
        for j in range(margin, X.shape[1] + margin):

            label = y[i-margin, j-margin]
            if label == 0:
                continue

            patch = padded[i-margin:i+margin+1, j-margin:j+margin+1]
            patch = patch.transpose(2,0,1)

            label_idx = label - 1

            class_samples.setdefault(label_idx, []).append(patch)

    X_out, y_out = [], []

    for cls, samples in class_samples.items():
        if len(samples) > limit_per_class:
            samples = random.sample(samples, limit_per_class)

        for s in samples:
            X_out.append(s)
            y_out.append(cls)

    return np.array(X_out), np.array(y_out)

print("Creating patches...")
X_data, y_data = create_samples(X, y, PATCH_SIZE, TRAIN_PER_CLASS)

# ================= SPLIT =================

indices = np.arange(len(X_data))
np.random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx, test_idx = indices[:split], indices[split:]

X_train, y_train = X_data[train_idx], y_data[train_idx]
X_test, y_test = X_data[test_idx], y_data[test_idx]

num_classes = len(np.unique(y_data))
class_names = [f"Class_{i}" for i in range(num_classes)]

# ================= DATASET =================

class HSIDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx])

train_loader = DataLoader(HSIDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(HSIDataset(X_test, y_test), batch_size=BATCH_SIZE)

# ================= CNN =================

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv3 = nn.Conv2d(3,16,3,padding=1)
        self.conv5 = nn.Conv2d(3,16,5,padding=2)
        self.conv7 = nn.Conv2d(3,16,7,padding=3)

        self.block1 = nn.Sequential(
            nn.Conv2d(48,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(256,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

    def forward(self,x):
        x = torch.cat([
            F.relu(self.conv3(x)),
            F.relu(self.conv5(x)),
            F.relu(self.conv7(x))
        ], dim=1)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        return x

model = CNN().to(DEVICE)
classifier = nn.Linear(256, num_classes).to(DEVICE)

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(classifier.parameters()),
    lr=0.0003
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# ================= TRAIN CNN =================

print("Training CNN...")
for epoch in range(CNN_EPOCHS):
    model.train()
    classifier.train()

    for x,yb in train_loader:
        x,yb = x.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad()

        feat = model(x)
        pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
        out = classifier(pooled)

        loss = criterion(out,yb)
        loss.backward()
        optimizer.step()

print("✅ CNN Done")

# ================= FEATURE EXTRACTION =================

def extract(loader):
    Xf, yf = [], []

    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)

            feat = model(xb)
            pooled = F.adaptive_avg_pool2d(feat,1).view(xb.size(0), -1)

            Xf.append(pooled.cpu().numpy())
            yf.extend(yb.numpy())

    return np.vstack(Xf).astype(np.float32), np.array(yf, dtype=np.uint32)

print("Extracting features...")
X_train_tm, y_train_tm = extract(train_loader)
X_test_tm,  y_test_tm  = extract(test_loader)

# ================= BINARIZATION =================

threshold = np.mean(X_train_tm, axis=0)
X_train_tm = (X_train_tm > threshold).astype(np.uint32)
X_test_tm  = (X_test_tm > threshold).astype(np.uint32)

# ================= TM =================

tm = TMClassifier(
    number_of_clauses=CLAUSES,
    T=T,
    s=S,
    weighted_clauses=True,
    feature_negation=True,
    platform="CPU"
)

print("Training TM...")

for epoch in range(TM_EPOCHS):
    tm.fit(X_train_tm, y_train_tm, epochs=1)
    preds = tm.predict(X_test_tm)
    acc = accuracy_score(y_test_tm, preds)
    print(f"Epoch {epoch+1} → {acc:.4f}")

print("🔥 FINAL ACC:", acc)

# ================= SAMPLE SELECTION =================

preds = tm.predict(X_test_tm)

clause_outputs = tm.transform(X_test_tm).reshape(
    X_test_tm.shape[0], num_classes, tm.number_of_clauses
)

scores = clause_outputs.sum(axis=2)

correct, wrong, ambiguous = [], [], []

for i in range(len(X_test_tm)):
    pred = preds[i]
    true = y_test_tm[i]

    sorted_scores = np.sort(scores[i])
    margin = sorted_scores[-1] - sorted_scores[-2]

    if pred == true:
        correct.append((i, margin))
    else:
        wrong.append((i, margin))

    if margin < 5:
        ambiguous.append((i, margin))

correct = sorted(correct, key=lambda x: -x[1])[:3]
wrong = sorted(wrong, key=lambda x: x[1])[:3]
ambiguous = sorted(ambiguous, key=lambda x: x[1])[:3]

# ================= HEATMAP =================

def generate_heatmap(idx, tag):
    os.makedirs("outputs", exist_ok=True)

    pred = preds[idx]
    true = y_test_tm[idx]

    print(f"[{tag.upper()}] idx={idx} → Pred: {class_names[pred]} | True: {class_names[true]}")

    img = X_test[idx][0]
    heatmap = cv2.resize(img, (64,64))
    heatmap = cv2.normalize(heatmap, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    img_color = cv2.cvtColor((heatmap).astype(np.uint8), cv2.COLOR_GRAY2BGR)

    overlay = cv2.addWeighted(img_color, 0.6, heatmap_color, 0.4, 0)

    cv2.imwrite(f"outputs/{tag}_{idx}_overlay.png", overlay)

# ================= RUN =================

print("\nGenerating interpretation...\n")

for i,_ in correct:
    generate_heatmap(i, "correct")

for i,_ in wrong:
    generate_heatmap(i, "wrong")

for i,_ in ambiguous:
    generate_heatmap(i, "ambiguous")

Loading Indian Pines dataset...
Creating patches...
Training CNN...
✅ CNN Done
Extracting features...
Training TM...
Epoch 1 → 0.7973
Epoch 2 → 0.8320
Epoch 3 → 0.8475
Epoch 4 → 0.8320
Epoch 5 → 0.8301
Epoch 6 → 0.8398
Epoch 7 → 0.8398
Epoch 8 → 0.8456
Epoch 9 → 0.8398
Epoch 10 → 0.8475
Epoch 11 → 0.8417
Epoch 12 → 0.8436
Epoch 13 → 0.8475
Epoch 14 → 0.8436
Epoch 15 → 0.8456
Epoch 16 → 0.8456
Epoch 17 → 0.8456
Epoch 18 → 0.8417
Epoch 19 → 0.8436
Epoch 20 → 0.8475
Epoch 21 → 0.8494
Epoch 22 → 0.8456
Epoch 23 → 0.8456
Epoch 24 → 0.8475
Epoch 25 → 0.8456
Epoch 26 → 0.8436
Epoch 27 → 0.8494
Epoch 28 → 0.8494
Epoch 29 → 0.8514
Epoch 30 → 0.8494
Epoch 31 → 0.8514
Epoch 32 → 0.8494
Epoch 33 → 0.8494
Epoch 34 → 0.8514
Epoch 35 → 0.8456
Epoch 36 → 0.8494
Epoch 37 → 0.8494
Epoch 38 → 0.8533
Epoch 39 → 0.8456
Epoch 40 → 0.8494
Epoch 41 → 0.8514
Epoch 42 → 0.8494
Epoch 43 → 0.8494
Epoch 44 → 0.8475
Epoch 45 → 0.8514
Epoch 46 → 0.8456
Epoch 47 → 0.8475
Epoch 48 → 0.8494
Epoch 49 → 0.8475
Epoch 50 →

/tmp/ipykernel_2674201/309519459.py:271: RuntimeWarning: overflow encountered in scalar negative
  correct = sorted(correct, key=lambda x: -x[1])[:3]


In [6]:
# ==========================================================
# 🚀 FINAL PIPELINE → CNN + TM + FULL INTERPRETATION
# ==========================================================

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from glob import glob
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from tmu.models.classification.vanilla_classifier import TMClassifier

# ================= CONFIG =================

BASE_DIR = "sample_dataset"
IMAGE_SIZE = 64
TRAIN_PER_CLASS = 121

BATCH_SIZE = 64
CNN_EPOCHS = 40

CLAUSES = 4000
T = 2500
S = 6.0
TM_EPOCHS = 50

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= DATA =================

train_samples, test_samples = [], []
classes = sorted(os.listdir(BASE_DIR))

for label, cname in enumerate(classes):
    class_path = os.path.join(BASE_DIR, cname)
    if not os.path.isdir(class_path):
        continue

    paths = glob(os.path.join(class_path, "*"))
    random.shuffle(paths)

    train_paths = paths[:TRAIN_PER_CLASS]
    test_paths  = paths[TRAIN_PER_CLASS:]

    train_samples += [(p, label) for p in train_paths]
    test_samples  += [(p, label) for p in test_paths]

num_classes = len(classes)

# ================= DATASET =================

class ImageDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path)
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2,0,1))
        return torch.tensor(img), torch.tensor(label)

train_loader = DataLoader(ImageDataset(train_samples), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(ImageDataset(test_samples), batch_size=BATCH_SIZE)

# ================= CNN =================

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv3 = nn.Conv2d(3,16,3,padding=1)
        self.conv5 = nn.Conv2d(3,16,5,padding=2)
        self.conv7 = nn.Conv2d(3,16,7,padding=3)

        self.block1 = nn.Sequential(nn.Conv2d(48,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.block2 = nn.Sequential(nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.block3 = nn.Sequential(nn.Conv2d(128,256,3,padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.block4 = nn.Sequential(nn.Conv2d(256,256,3,padding=1), nn.ReLU())

    def forward(self,x):
        x = torch.cat([
            F.relu(self.conv3(x)),
            F.relu(self.conv5(x)),
            F.relu(self.conv7(x))
        ], dim=1)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        return x

model = CNN().to(DEVICE)
classifier = nn.Linear(256, num_classes).to(DEVICE)

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(classifier.parameters()), lr=0.0003
)
criterion = nn.CrossEntropyLoss()

# ================= TRAIN CNN =================

for epoch in range(CNN_EPOCHS):
    model.train()
    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        feat = model(x)
        pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
        out = classifier(pooled)

        loss = criterion(out,y)
        loss.backward()
        optimizer.step()

print("✅ CNN DONE")

# ================= FEATURE EXTRACTION =================

def extract(loader):
    X, y = [], []
    model.eval()

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            feat = model(xb)
            pooled = F.adaptive_avg_pool2d(feat,1).view(xb.size(0), -1)

            X.append(pooled.cpu().numpy())
            y.extend(yb.numpy())

    return np.vstack(X), np.array(y)

X_train_tm, y_train_tm = extract(train_loader)
X_test_tm, y_test_tm   = extract(test_loader)

# 🔥 FIX: convert labels to uint32
y_train_tm = y_train_tm.astype(np.uint32)
y_test_tm  = y_test_tm.astype(np.uint32)

# ================= BINARIZE =================

thr = np.mean(X_train_tm, axis=0)
X_train_tm = (X_train_tm > thr).astype(np.uint32)
X_test_tm  = (X_test_tm > thr).astype(np.uint32)

# ================= TM =================

tm = TMClassifier(
    number_of_clauses=CLAUSES,
    T=T,
    s=S,
    weighted_clauses=True,
    feature_negation=True,
    platform="CPU"
)

for epoch in range(TM_EPOCHS):
    tm.fit(X_train_tm, y_train_tm, epochs=1)
    preds = tm.predict(X_test_tm)
    acc = accuracy_score(y_test_tm, preds)
    print(f"Epoch {epoch+1}: {acc:.4f}")

print("🔥 FINAL ACC:", acc)

# ================= ANALYSIS =================

preds = tm.predict(X_test_tm)

clause_outputs = tm.transform(X_test_tm).reshape(
    X_test_tm.shape[0], num_classes, CLAUSES
)

scores = clause_outputs.sum(axis=2)

correct, wrong, ambiguous = [], [], []

for i in range(len(X_test_tm)):
    pred = preds[i]
    true = y_test_tm[i]

    sorted_scores = np.sort(scores[i])
    margin = sorted_scores[-1] - sorted_scores[-2]

    if pred == true:
        correct.append((i, margin))
    else:
        wrong.append((i, margin))

    if margin < 5:
        ambiguous.append((i, margin))

correct   = sorted(correct, key=lambda x:-x[1])[:3]
wrong     = sorted(wrong, key=lambda x:x[1])[:3]
ambiguous = sorted(ambiguous, key=lambda x:x[1])[:3]

# ================= INTERPRETATION =================

def interpret(idx, tag):
    os.makedirs("outputs", exist_ok=True)

    pred = preds[idx]
    true = y_test_tm[idx]

    print(f"\n[{tag}] INDEX: {idx}")
    print(f"TRUE: {true} | PRED: {pred}")

    sorted_scores = np.sort(scores[idx])
    margin = sorted_scores[-1] - sorted_scores[-2]
    print("CONFIDENCE:", margin)

    active = np.where(X_test_tm[idx]==1)[0]
    print("ACTIVE FEATURES:", len(active))

    # ===== Heatmap =====
    img, _ = test_loader.dataset[idx]
    img_np = img[0].numpy()

    heatmap = np.zeros((IMAGE_SIZE, IMAGE_SIZE))

    for f in active:
        x = f % IMAGE_SIZE
        y = f // IMAGE_SIZE
        if x < IMAGE_SIZE and y < IMAGE_SIZE:
            heatmap[y,x] += 1

    heatmap = cv2.GaussianBlur(heatmap,(7,7),0)
    heatmap = cv2.normalize(heatmap,None,0,255,cv2.NORM_MINMAX).astype(np.uint8)

    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    img_color = cv2.cvtColor((img_np*255).astype(np.uint8), cv2.COLOR_GRAY2BGR)

    overlay = cv2.addWeighted(img_color,0.6,heatmap_color,0.4,0)

    path = f"outputs/{tag}_{idx}.png"
    cv2.imwrite(path, overlay)

    print("Saved:", path)

# ================= RUN =================

print("\n===== INTERPRETATION RESULTS =====")

for i,_ in correct:
    interpret(i, "CORRECT")

for i,_ in wrong:
    interpret(i, "WRONG")

for i,_ in ambiguous:
    interpret(i, "AMBIGUOUS")

✅ CNN DONE
Epoch 1: 0.6126
Epoch 2: 0.6859
Epoch 3: 0.6702
Epoch 4: 0.7382
Epoch 5: 0.7277
Epoch 6: 0.7277
Epoch 7: 0.7068
Epoch 8: 0.7644
Epoch 9: 0.7487
Epoch 10: 0.7749
Epoch 11: 0.8063
Epoch 12: 0.7487
Epoch 13: 0.8010
Epoch 14: 0.7749
Epoch 15: 0.7487
Epoch 16: 0.7749
Epoch 17: 0.8010
Epoch 18: 0.7801
Epoch 19: 0.7696
Epoch 20: 0.8063
Epoch 21: 0.7801
Epoch 22: 0.8115
Epoch 23: 0.7906
Epoch 24: 0.8010
Epoch 25: 0.8063
Epoch 26: 0.8115
Epoch 27: 0.7696
Epoch 28: 0.7801
Epoch 29: 0.8063
Epoch 30: 0.8115
Epoch 31: 0.8010
Epoch 32: 0.8168
Epoch 33: 0.8168
Epoch 34: 0.8115
Epoch 35: 0.8063
Epoch 36: 0.8115
Epoch 37: 0.8220
Epoch 38: 0.8272
Epoch 39: 0.8063
Epoch 40: 0.7906
Epoch 41: 0.8220
Epoch 42: 0.7853
Epoch 43: 0.7958
Epoch 44: 0.8272
Epoch 45: 0.8063
Epoch 46: 0.8168
Epoch 47: 0.8325
Epoch 48: 0.8220
Epoch 49: 0.8168
Epoch 50: 0.8220
🔥 FINAL ACC: 0.8219895287958116

===== INTERPRETATION RESULTS =====

[CORRECT] INDEX: 160
TRUE: 9 | PRED: 9
CONFIDENCE: 0
ACTIVE FEATURES: 83
Saved:

/tmp/ipykernel_2674201/3118810659.py:204: RuntimeWarning: overflow encountered in scalar negative
  correct   = sorted(correct, key=lambda x:-x[1])[:3]


In [9]:
# ==========================================================
# 🚀 EuroSAT → CNN → TM → FULL INTERPRETATION (FIXED)
# ==========================================================

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from glob import glob
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from tmu.models.classification.vanilla_classifier import TMClassifier

# ================= CONFIG =================

BASE_DIR = "EuroSAT"
IMAGE_SIZE = 64
TRAIN_PER_CLASS = 121

BATCH_SIZE = 64
CNN_EPOCHS = 40

CLAUSES = 8000     # ✅ PER CLASS (IMPORTANT)
T = 3000
S = 7.0
TM_EPOCHS = 50

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= DATA =================

train_samples, test_samples = [], []
classes = sorted(os.listdir(BASE_DIR))

for label, cname in enumerate(classes):
    path = os.path.join(BASE_DIR, cname)
    if not os.path.isdir(path):
        continue

    imgs = glob(os.path.join(path, "*"))
    random.shuffle(imgs)

    train = imgs[:TRAIN_PER_CLASS]
    test  = imgs[TRAIN_PER_CLASS:]

    train_samples += [(p, label) for p in train]
    test_samples  += [(p, label) for p in test]

num_classes = len(classes)

# ================= DATASET =================

class EuroSATDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = cv2.imread(path)
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2,0,1))

        return torch.tensor(img), torch.tensor(label)

train_loader = DataLoader(EuroSATDataset(train_samples), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(EuroSATDataset(test_samples), batch_size=BATCH_SIZE)

# ================= CNN =================

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv3 = nn.Conv2d(3,16,3,padding=1)
        self.conv5 = nn.Conv2d(3,16,5,padding=2)
        self.conv7 = nn.Conv2d(3,16,7,padding=3)

        self.block1 = nn.Sequential(nn.Conv2d(48,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.block2 = nn.Sequential(nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.block3 = nn.Sequential(nn.Conv2d(128,256,3,padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.block4 = nn.Sequential(nn.Conv2d(256,256,3,padding=1), nn.ReLU())

    def forward(self,x):
        x = torch.cat([
            F.relu(self.conv3(x)),
            F.relu(self.conv5(x)),
            F.relu(self.conv7(x))
        ], dim=1)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        return x

model = CNN().to(DEVICE)
classifier = nn.Linear(256, num_classes).to(DEVICE)

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(classifier.parameters()), lr=0.0003
)
criterion = nn.CrossEntropyLoss()

# ================= TRAIN CNN =================

print("Training CNN...")

for epoch in range(CNN_EPOCHS):
    model.train()
    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        feat = model(x)
        pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
        out = classifier(pooled)

        loss = criterion(out,y)
        loss.backward()
        optimizer.step()

print("✅ CNN DONE")

# ================= FEATURE EXTRACTION =================

def extract(loader):
    X, y = [], []
    model.eval()

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            feat = model(xb)
            pooled = F.adaptive_avg_pool2d(feat,1).view(xb.size(0), -1)

            X.append(pooled.cpu().numpy())
            y.extend(yb.numpy())

    return np.vstack(X).astype(np.float32), np.array(y, dtype=np.uint32)

X_train, y_train = extract(train_loader)
X_test,  y_test  = extract(test_loader)

# ================= BETTER BINARIZATION =================

thr = np.percentile(X_train, 60, axis=0)

X_train = (X_train > thr).astype(np.uint32)
X_test  = (X_test > thr).astype(np.uint32)

# ✅ FEATURE EXPANSION (VERY IMPORTANT)
X_train = np.concatenate([X_train, 1 - X_train], axis=1)
X_test  = np.concatenate([X_test, 1 - X_test], axis=1)

# ================= TM =================

tm = TMClassifier(
    number_of_clauses=CLAUSES,
    T=T,
    s=S,
    weighted_clauses=True,
    feature_negation=True,
    platform="CPU"
)

print("Training TM...")

for epoch in range(TM_EPOCHS):
    tm.fit(X_train, y_train, epochs=1)

    preds = tm.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"Epoch {epoch+1}: {acc:.4f}")

print("🔥 FINAL ACC:", acc)

# ================= ANALYSIS =================

preds = tm.predict(X_test)
clause_outputs_raw = tm.transform(X_test)

n_samples = clause_outputs_raw.shape[0]

# ✅ CORRECT SCORE COMPUTATION (NO RESHAPE)
scores = np.zeros((n_samples, num_classes))

for c in range(num_classes):
    start = c * CLAUSES
    end   = (c + 1) * CLAUSES
    scores[:, c] = clause_outputs_raw[:, start:end].sum(axis=1)

# ================= SAMPLE SELECTION =================

correct, wrong, ambiguous = [], [], []

for i in range(n_samples):
    pred = preds[i]
    true = y_test[i]

    sorted_scores = np.sort(scores[i])
    margin = sorted_scores[-1] - sorted_scores[-2]

    if pred == true:
        correct.append((i, margin))
    else:
        wrong.append((i, margin))

    if margin < 5:
        ambiguous.append((i, margin))

correct   = sorted(correct, key=lambda x:-x[1])[:3]
wrong     = sorted(wrong, key=lambda x:x[1])[:3]
ambiguous = sorted(ambiguous, key=lambda x:x[1])[:3]

# ================= INTERPRETATION =================

def interpret(idx, tag):
    os.makedirs("outputs", exist_ok=True)

    pred = preds[idx]
    true = y_test[idx]

    print("\n==============================")
    print(f"[{tag}] INDEX: {idx}")
    print(f"TRUE: {true} | PRED: {pred}")

    sorted_scores = np.sort(scores[idx])
    margin = sorted_scores[-1] - sorted_scores[-2]
    print("CONFIDENCE:", margin)

    img, _ = test_loader.dataset[idx]
    img_np = np.transpose(img.numpy(), (1,2,0))

    xb = img.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        feat = model(xb)[0]

    heatmap = feat.mean(dim=0).cpu().numpy()
    heatmap = cv2.resize(heatmap, (IMAGE_SIZE, IMAGE_SIZE))
    heatmap = cv2.normalize(heatmap, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    img_color = (img_np * 255).astype(np.uint8)

    overlay = cv2.addWeighted(img_color, 0.6, heatmap_color, 0.4, 0)

    path = f"outputs/{tag}_{idx}.png"
    cv2.imwrite(path, overlay)

    print("Saved:", path)

# ================= RUN =================

print("\n===== INTERPRETATION RESULTS =====")

for i,_ in correct:
    interpret(i, "CORRECT")

for i,_ in wrong:
    interpret(i, "WRONG")

for i,_ in ambiguous:
    interpret(i, "AMBIGUOUS")

Training CNN...
✅ CNN DONE
Training TM...
Epoch 1: 0.6177
Epoch 2: 0.6248
Epoch 3: 0.6630
Epoch 4: 0.6653
Epoch 5: 0.6718
Epoch 6: 0.6604
Epoch 7: 0.6762
Epoch 8: 0.6811
Epoch 9: 0.6829
Epoch 10: 0.6840
Epoch 11: 0.6796
Epoch 12: 0.6876
Epoch 13: 0.6853
Epoch 14: 0.6835
Epoch 15: 0.6789
Epoch 16: 0.6874
Epoch 17: 0.6730
Epoch 18: 0.6810
Epoch 19: 0.6825
Epoch 20: 0.6959
Epoch 21: 0.6906
Epoch 22: 0.6875
Epoch 23: 0.6902
Epoch 24: 0.6862
Epoch 25: 0.6899
Epoch 26: 0.6918
Epoch 27: 0.6884
Epoch 28: 0.6920
Epoch 29: 0.6899
Epoch 30: 0.6865
Epoch 31: 0.6836
Epoch 32: 0.6784
Epoch 33: 0.6841
Epoch 34: 0.6851
Epoch 35: 0.6815
Epoch 36: 0.6898
Epoch 37: 0.6848
Epoch 38: 0.6848
Epoch 39: 0.6799
Epoch 40: 0.6857
Epoch 41: 0.6874
Epoch 42: 0.6878
Epoch 43: 0.6827
Epoch 44: 0.6818
Epoch 45: 0.6808
Epoch 46: 0.6871
Epoch 47: 0.6810
Epoch 48: 0.6843
Epoch 49: 0.6831
Epoch 50: 0.6836
🔥 FINAL ACC: 0.6836370686312524

===== INTERPRETATION RESULTS =====

[CORRECT] INDEX: 8844
TRUE: 3 | PRED: 3
CONFIDEN